# Notebook 07: Train Multimodal Deep Learning Model (PyTorch)
**GlobalFlood AI: Global Multimodal Flood Prediction System (48-Hour Forecast)**

### Objective
Train the core **MultimodalFloodNet** architecture using PyTorch:
- **Branch 1:** Sentinel-1 SAR Dual-Pol CNN (128-d representation)
- **Branch 2:** Weather 168h Historical + 48h Forecast GRU (64-d representation)
- **Branch 3:** Topographic Feature MLP (32-d representation)
- **Branch 4:** Mask-Aware Fusion Layer (227-d input $\to$ 128 $\to$ 64 $\to$ 1 logit)
- Class-weighted Binary Cross-Entropy loss for class imbalance handling
- ReduceLROnPlateau learning rate scheduling & model checkpoint saving

### Inputs & Outputs
- **Inputs:** Train and validation samples
- **Outputs:** Model checkpoint saved to `models/checkpoints/multimodal_flood_net_best.pt`
- **Dependencies:** `torch`, `ml.src.train`, `ml.src.model`


In [ ]:
import json
import torch
from ml.src.train import train_multimodal_model, get_device
from app.core.config import settings

device = get_device()
print(f"Training on device: {device}")

# Load samples
with open(settings.PROCESSED_DATA_DIR / "train_samples.json") as f:
    train_samples = json.load(f)
with open(settings.PROCESSED_DATA_DIR / "val_samples.json") as f:
    val_samples = json.load(f)

# Train multimodal model
res = train_multimodal_model(
    train_samples=train_samples,
    val_samples=val_samples,
    epochs=5,
    lr=1e-3,
    batch_size=4,
    save_path=settings.MODEL_WEIGHTS_FILE,
    is_smoke_test=False
)

history = res["history"]
print(f"\nTraining complete! Best validation loss: {res['best_val_loss']:.4f}")
print("Final Epoch Train Loss:", history["train_loss"][-1])
print("Final Epoch Val Loss:  ", history["val_loss"][-1])
print("Final Epoch Val AUC:   ", history["val_auc"][-1])
print("\n[SUCCESS] MultimodalFloodNet model successfully trained and checkpoint saved.")
